# Chapter 8 — Distribution functions of spherical systems, and how to sample them

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/08_spherical_dfs.ipynb)

**Goal.** Chapter 7 used only velocity *moments*. Here we construct the full distribution function
$f(\mathbf x, \mathbf v)$ (the density of stars in the space of positions $\mathbf x$ and velocities $\mathbf v$; Chapter 6) of a spherical galaxy in equilibrium, draw stars from it, and show why a
simulation should start from a real DF rather than from moments.

**You will learn**
1. the **Eddington inversion**: the unique isotropic DF $f(\mathcal E)$ of a given density in a given potential, where
   $\mathcal E = -\Phi - \tfrac12 v^2$ is the relative energy per unit mass ($\Phi$ the potential, $v$ the speed; §8.1);
2. to check a DF by recomputing the density from it, and against AGAMA's `QuasiSpherical` DF;
3. to **sample** stars from a DF, by hand and with AGAMA, and to check the sample;
4. that a sample with the right velocity dispersions is **not** necessarily in equilibrium;
5. **anisotropic** DFs of Osipkov–Merritt type, and when a DF becomes negative (unphysical).

**Units.** $M_\odot$ (solar mass), kpc, km/s.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object
G = agama.G
TIME_UNIT_MYR = agama.getUnits()['time']

import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from scipy.special import gamma as Gamma

rng = np.random.default_rng(8)

## 8.1 The Eddington inversion

By the Jeans theorem (Chapter 6), a steady spherical system can have a DF that depends only on the
energy. It is convenient to use positive quantities: the **relative potential** $\Psi = -\Phi$ ($\Phi$ is the gravitational potential)
and the **relative energy** $\mathcal E = \Psi - \tfrac12 v^2$ ($v$ the speed); bound stars have $\mathcal E > 0$. Such a DF is
**isotropic** (anisotropy $\beta = 0$, Chapter 7), because it depends on the speed but not on the direction of the velocity.

The density is the integral of $f$ over velocities. With $d^3v = 4\pi v^2 dv$ and $v = \sqrt{2(\Psi - \mathcal E)}$:

$$ \rho(r) = 4\pi\int_0^{\Psi(r)} f(\mathcal E)\,\sqrt{2(\Psi - \mathcal E)}\, d\mathcal E , $$

where $\rho(r)$ is the mass density at radius $r$.

Since $\Psi(r)$ decreases monotonically, $\rho$ can be regarded as a function of $\Psi$, and this is an Abel
integral equation for $f$. Its solution (Eddington 1916; Binney & Tremaine §4.3.1) is

$$ f(\mathcal E) = \frac{1}{\sqrt 8\,\pi^2}\left[\int_0^{\mathcal E} \frac{d^2\rho}{d\Psi^2}\frac{d\Psi}{\sqrt{\mathcal E - \Psi}}
   + \frac{1}{\sqrt{\mathcal E}}\left(\frac{d\rho}{d\Psi}\right)_{\Psi=0}\right]. $$

The last term vanishes for all the models here (their $d\rho/d\Psi \to 0$ far away), so we drop it.

**Numerically:** tabulate $\Psi(r)$ and $\rho(r)$ on a grid of radii, fit a spline to $\ln\rho$ as a function of $\ln\Psi$,
and get $d^2\rho/d\Psi^2$ from the spline by the chain rule. The substitution $\Psi = \mathcal E - t^2$ turns
$d\Psi/\sqrt{\mathcal E - \Psi}$ into $2\,dt$, which removes the infinity at $\Psi = \mathcal E$.

Any profile works, so the function takes an AGAMA potential (for $\Psi$) and an AGAMA density (for $\rho$). The optional
argument `r_a` is used in §8.4.

In [ ]:
def eddington(pot, dens, r_a=np.inf):
    """Isotropic DF f(E) of the density `dens` in the potential `pot`, by Eddington's formula.
    With finite r_a, the Osipkov-Merritt DF f(Q) of §8.4 (dens is multiplied by 1 + r^2/r_a^2).
    Returns a function of the relative energy E (or Q)."""
    r = np.geomspace(1e-5, 1e6, 1500)                 # wide range: f(E) for weakly bound E needs large r
    pts = np.column_stack([r, 0 * r, 0 * r])
    psi = -pot.potential(pts)
    rho = dens.density(pts) * (1 + r**2 / r_a**2)
    spl = CubicSpline(np.log(psi[::-1]), np.log(rho[::-1]))   # spline needs increasing x: reverse arrays
    def d2rho_dpsi2(p):
        # rho = exp(s(x)), x = ln psi  =>  d2rho/dpsi2 = rho/psi^2 (s'' + s'^2 - s')
        x = np.log(p)
        s1, s2 = spl(x, 1), spl(x, 2)
        return np.exp(spl(x)) / p**2 * (s2 + s1**2 - s1)
    psi_min = psi[-1]                                 # outermost tabulated point: f = 0 below it
    def f(E):
        if E <= psi_min:
            return 0.0
        integral = quad(lambda t: 2 * d2rho_dpsi2(E - t * t), 0, np.sqrt(E - psi_min), limit=200)[0]
        return integral / (np.sqrt(8) * np.pi**2)
    return f

M, b = 1e10, 1.0
plummer = agama.Potential(type='Plummer', mass=M, scaleRadius=b)    # potential and density in one object
f_plummer = eddington(plummer, plummer)
Psi0 = G * M / b                                     # relative potential at the centre = most bound energy

# For the Plummer sphere the integral can be done analytically: f is a power law of the energy
f_exact = lambda E: 24 * np.sqrt(2) / (7 * np.pi**3) * b**2 / (G**5 * M**4) * E**3.5

E_grid = np.linspace(0.01, 0.99, 50) * Psi0
f_num = np.array([f_plummer(E) for E in E_grid])
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy(E_grid / Psi0, f_exact(E_grid), 'k-', lw=3, alpha=0.3, label='exact $\\propto \\mathcal{E}^{7/2}$')
ax.semilogy(E_grid / Psi0, f_num, 'C0--', label='Eddington, numerical')
ax.set_xlabel('$\\mathcal{E}/\\Psi(0)$'); ax.set_ylabel('$f$ [$M_\\odot$ / (kpc km/s)$^3$]'); ax.legend()
plt.show()
print(f'max relative error of f: {np.max(np.abs(f_num / f_exact(E_grid) - 1)):.1e}')
assert np.max(np.abs(f_num / f_exact(E_grid) - 1)) < 1e-6

**Figure 8.1.** Isotropic distribution function of the Plummer sphere against relative energy $\mathcal E/\Psi(0)$: the exact power law $f \propto \mathcal E^{7/2}$ (grey) and the numerical Eddington inversion (dashed).

`f` is a mass density in 6-d phase space: $M_\odot$ per kpc$^3$ per (km/s)$^3$.

**Check 1: the DF gives back the density.** Integrate $f$ over velocities at several radii.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')                  # quad warns about round-off near the integrable end points
    for r in [0.1, 1.0, 5.0, 20.0]:
        Psi = -plummer.potential([r, 0, 0])
        rho_from_f = 4 * np.pi * quad(lambda E: f_plummer(E) * np.sqrt(2 * (Psi - E)), 0, Psi, limit=200)[0]
        err = rho_from_f / plummer.density([r, 0, 0]) - 1
        print(f'r = {r:5.1f} kpc: integral of f d^3v / rho - 1 = {err:.1e}')
        assert abs(err) < 1e-3

**Check 2: AGAMA.** AGAMA's `QuasiSpherical` DF (which we used in Chapter 7) is constructed by the same
inversion. AGAMA DFs are functions of the **actions** $\mathbf J = (J_r, J_z, J_\phi)$ (radial, vertical and azimuthal integrals of
motion in kpc km/s, Chapter 9), not of the energy: to evaluate it at a
phase-space point we first compute the actions with an `ActionFinder`. A second model, the Hernquist profile
(AGAMA type `Dehnen` with `gamma=1`), has a central cusp $\rho \propto 1/r$, unlike the Plummer core.

In [ ]:
xv_test = np.column_stack([rng.uniform(0.2, 5, (8, 3)), rng.normal(0, 60, (8, 3))])   # 8 random phase-space points
for name, model in [('Plummer', plummer), ('Hernquist', agama.Potential(type='Dehnen', gamma=1, mass=M, scaleRadius=b))]:
    df = agama.DistributionFunction(type='QuasiSpherical', potential=model, density=model)
    f_agama = df(agama.ActionFinder(model)(xv_test))
    E_rel = -(model.potential(xv_test[:, :3]) + 0.5 * np.sum(xv_test[:, 3:]**2, axis=1))
    f_mine = np.array([eddington(model, model)(E) for E in E_rel])
    bound = E_rel > 0                                 # unbound points have f = 0 in both
    print(f'{name:9s}: {bound.sum()} bound points, max |f_AGAMA / f_Eddington - 1| = '
          f'{np.max(np.abs(f_agama[bound] / f_mine[bound] - 1)):.1e}')
    assert np.max(np.abs(f_agama[bound] / f_mine[bound] - 1)) < 1e-3

## 8.2 Sampling stars from the DF

To set up a simulation, or a mock catalogue, we draw $N$ stars with probability proportional to $f$.
For an isotropic spherical model this splits into simple steps:

1. **radius**: the fraction of mass inside $r$ is $X = M(r)/M$, where $M$ is the total mass. Draw $X$ uniformly in $[0, 1]$ and
   invert; for the Plummer sphere of scale radius $b$, $\rho \propto (1 + r^2/b^2)^{-5/2}$,
   $X = r^3/(r^2+b^2)^{3/2}$ gives $r = b/\sqrt{X^{-2/3} - 1}$ (*inverse transform sampling*);
2. **speed** at that radius: the probability of speed $v$ is $\propto v^2 f(\Psi - v^2/2)$ between 0 and the escape
   speed $v_{\rm esc} = \sqrt{2\Psi}$. With $q = v/v_{\rm esc}$ and $f \propto \mathcal E^{7/2}$ this is $p(q) \propto q^2(1-q^2)^{7/2}$. We draw it by
   **rejection**: propose $q$ uniformly and $y$ uniformly below the maximum of $p$, and accept if $y < p(q)$;
3. **directions** of position and velocity: uniformly on the sphere (polar angle $\theta$ with uniform $\cos\theta$ in $[-1, 1]$,
   azimuth $\phi$ uniform in $[0, 2\pi)$).

In [ ]:
def random_directions(n):
    """n unit vectors uniformly distributed on the sphere."""
    cos_t, phi = rng.uniform(-1, 1, n), rng.uniform(0, 2 * np.pi, n)
    sin_t = np.sqrt(1 - cos_t**2)
    return np.column_stack([sin_t * np.cos(phi), sin_t * np.sin(phi), cos_t])

def sample_plummer(n):
    r = b / np.sqrt(rng.random(n)**(-2 / 3) - 1)
    p_max = (2 / 9) * (7 / 9)**3.5                    # maximum of q^2 (1-q^2)^3.5, at q^2 = 2/9
    q = np.empty(n)
    todo = np.arange(n)                               # indices of stars still without a speed
    while todo.size > 0:
        q_try, y = rng.random(todo.size), rng.random(todo.size) * p_max
        accept = y < q_try**2 * (1 - q_try**2)**3.5
        q[todo[accept]] = q_try[accept]
        todo = todo[~accept]                          # try again for the rejected ones
    v = q * np.sqrt(2 * G * M / np.sqrt(r**2 + b**2))
    return np.hstack([r[:, None] * random_directions(n), v[:, None] * random_directions(n)])

N = 100000
xv_hand = sample_plummer(N)
df_plummer = agama.DistributionFunction(type='QuasiSpherical', potential=plummer, density=plummer)
xv_agama, m_agama = agama.GalaxyModel(plummer, df_plummer).sample(N)    # AGAMA: one line
print(f'mass of the DF (df.totalMass()): {df_plummer.totalMass() / M - 1:+.1e} relative to M')
print(f'sum of the sampled particle masses: {m_agama.sum() / M - 1:+.2%} relative to M')

AGAMA gives every particle the same mass. Their sum is AGAMA's Monte Carlo estimate of the total mass made
while sampling, so it is off by a few tenths of a per cent (it changes from run to run). The DF itself has the
right mass, as `df.totalMass()` shows; rescale the particle masses if you need the exact total.

Three checks on each sample: the density profile, the radial velocity dispersion against the
Jeans-equation result of Chapter 7, $\sigma_r^2 = GM/(6\sqrt{r^2+b^2})$ ($G$ is the gravitational constant), and the virial theorem
$2K + W = 0$ (Chapter 1), where $K$ is the total kinetic energy of the stars and $W$ their potential energy, $W = -3\pi GM^2/(32 b)$
for the Plummer sphere.

In [ ]:
edges = np.geomspace(0.05, 20, 25)
rc = np.sqrt(edges[1:] * edges[:-1])
W = -3 * np.pi * G * M**2 / (32 * b)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for name, xv in [('by hand', xv_hand), ('AGAMA', xv_agama)]:
    r = np.linalg.norm(xv[:, :3], axis=1)
    v_r = np.sum(xv[:, :3] * xv[:, 3:], axis=1) / r
    n = np.histogram(r, edges)[0]
    rho_s = n * (M / N) / (4 / 3 * np.pi * np.diff(edges**3))
    sr2 = np.histogram(r, edges, weights=v_r**2)[0] / n
    ax[0].loglog(rc, rho_s, 'o', ms=4, label=name)
    ax[1].semilogx(rc, np.sqrt(sr2), 'o', ms=4, label=name)
    K = 0.5 * (M / N) * np.sum(xv[:, 3:]**2)
    print(f'{name:8s}: 2K/|W| = {2 * K / abs(W):.4f}')
    assert abs(2 * K / abs(W) - 1) < 0.01
ax[0].loglog(rc, plummer.density(np.column_stack([rc, 0 * rc, 0 * rc])), 'k-', label='model')
ax[1].semilogx(rc, np.sqrt(G * M / (6 * np.sqrt(rc**2 + b**2))), 'k-', label='Jeans (Ch. 7)')
ax[0].set_xlabel('$r$ [kpc]'); ax[0].set_ylabel('$\\rho$ [$M_\\odot$/kpc$^3$]'); ax[0].legend()
ax[1].set_xlabel('$r$ [kpc]'); ax[1].set_ylabel('$\\sigma_r$ [km/s]'); ax[1].legend()
plt.tight_layout(); plt.show()

**Figure 8.2.** Two samples of $10^5$ stars drawn from the Plummer DF, by hand and with AGAMA. Left: density profile against the model; right: radial velocity dispersion $\sigma_r$ against the Jeans-equation result of Chapter 7.

Both samples follow the model (Figure 8.2); the scatter at large radii is Poisson noise (few stars per bin). The virial
ratio is 1 within the statistical error, which for $10^5$ stars is about $0.2\%$.

## 8.3 Right moments, wrong equilibrium

A popular shortcut for initial conditions is to place stars at the right radii and give them Gaussian
velocities with the dispersion from the Jeans equation. The density and all the second moments are then
correct, so the Jeans equations are satisfied. Is the result in equilibrium?

Test it: integrate both samples in the (fixed) Plummer potential for 30 central oscillation periods
$T_0 = 2\pi/\sqrt{GM/b^3}$, and follow the radii containing 10%, 50% and 90% of the stars (**Lagrange radii**).
For the model they are $b/\sqrt{X^{-2/3}-1}$ = 0.52, 1.30 and 3.71 kpc. A sample in equilibrium keeps them.

In [ ]:
Nt = 20000
xv_edd = xv_hand[:Nt]
r_edd = np.linalg.norm(xv_edd[:, :3], axis=1)
sigma = np.sqrt(G * M / (6 * np.sqrt(r_edd**2 + b**2)))         # Jeans dispersion at each star's radius
xv_gauss = np.hstack([xv_edd[:, :3], sigma[:, None] * rng.normal(size=(Nt, 3))])   # same positions

E_gauss = plummer.potential(xv_gauss[:, :3]) + 0.5 * np.sum(xv_gauss[:, 3:]**2, axis=1)
print(f'Gaussian velocities: {np.mean(E_gauss > 0):.2%} of the stars are faster than the escape speed')

T0 = 2 * np.pi / np.sqrt(G * M / b**3)
X_lagr = np.array([0.1, 0.5, 0.9])
lagr_model = b / np.sqrt(X_lagr**(-2 / 3) - 1)
fig, ax = plt.subplots(figsize=(8, 4.5))
lagr_end = {}
for name, ic, ls in [('Eddington DF', xv_edd, '-'), ('Gaussian, Jeans $\\sigma$', xv_gauss, '--')]:
    orbits = agama.orbit(potential=plummer, ic=ic, time=30 * T0, trajsize=61, verbose=False)
    t, traj = orbits[0, 0], np.stack(orbits[:, 1])   # traj: (stars, 61 snapshots, 6)
    lagr = np.quantile(np.linalg.norm(traj[:, :, :3], axis=2), X_lagr, axis=0)   # (3, 61): quantiles over stars
    lagr_end[name] = lagr[:, -1]
    for k in range(3):
        ax.plot(t / T0, lagr[k] / lagr_model[k], ls, color=f'C{k}', label=f'{name}, {X_lagr[k]:.0%}')
ax.set_xlabel('$t / T_0$'); ax.set_ylabel('Lagrange radius / model value'); ax.set_ylim(0.9, 1.3)
ax.legend(fontsize=8, ncol=2, loc='upper right')
plt.show()

for name, lr in lagr_end.items():
    print(f'{name:26s} Lagrange radii after 30 T0 / model: {np.round(lr / lagr_model, 3)}')
assert np.all(np.abs(lagr_end['Eddington DF'] / lagr_model - 1) < 0.03)
assert lagr_end['Gaussian, Jeans $\\sigma$'][2] / lagr_model[2] > 1.05

**Figure 8.3.** Lagrange radii containing 10%, 50% and 90% of the stars, divided by their model values, during 30 central periods $T_0$ in the fixed Plummer potential. Solid: the Eddington-DF sample; dashed: the same positions with Gaussian velocities of the Jeans dispersion.

In Figure 8.3 the Eddington sample stays put. The Gaussian sample settles, within a few periods, into a different
equilibrium: the 90% radius grows by about 10%, the 10% radius shrinks by a few per cent, and some stars escape. The true velocity
distribution at each radius is not Gaussian: it is cut off at the escape speed and has a different shape.
Only a DF that depends on the integrals of motion is guaranteed to be steady (the Jeans theorem); the
Jeans equations constrain only its moments.

(Our test uses a fixed potential. In a self-gravitating $N$-body simulation the potential would change
too, but the conclusion is the same.)

## 8.4 Anisotropic DFs: Osipkov–Merritt models

To make a model with radially biased velocities, let the DF depend on the angular momentum $L$ as well as
the energy ($L = |\mathbf x \times \mathbf v|$, the magnitude of the angular momentum per unit mass). Osipkov (1979) and Merritt (1985) used the combination

$$ Q = \mathcal E - \frac{L^2}{2 r_a^2}, \qquad f = f(Q). $$

where $r_a$ is the **anisotropy radius**. Stars at $r \ll r_a$ are nearly isotropic; far outside $r_a$ the orbits become radial, with
$\beta(r) = r^2/(r^2 + r_a^2)$ (Chapter 7). Remarkably, the Eddington formula still applies: replace $\rho$ by
$\rho_Q = (1 + r^2/r_a^2)\,\rho$ and $\mathcal E$ by $Q$. Our function does this when given `r_a`.

A DF must be **non-negative**. The more radial the orbits, the harder it is to build the central density
from them, and for small $r_a$ the inversion returns $f < 0$ near the centre. For the Plummer sphere we can
find the limit analytically. With $\Psi_0 = \Psi(0)$, the relative potential at the centre, and $x = \Psi/\Psi_0$,
$\rho_Q \propto (1 - k)\,x^5 + k\,x^3$ where $k = b^2/r_a^2$. The Eddington formula turns $x^n$ into
$\Gamma(n+1)/\Gamma(n - \tfrac12)\;(Q/\Psi_0)^{n - 3/2}$, where $\Gamma$ is the gamma function, so at the centre ($Q = \Psi_0$)

$$ f \propto (1-k)\frac{\Gamma(6)}{\Gamma(9/2)} + k\,\frac{\Gamma(4)}{\Gamma(5/2)} \ge 0
   \quad\Longleftrightarrow\quad k \le \frac{16}{9} \quad\Longleftrightarrow\quad r_a \ge \tfrac34\, b . $$

In [ ]:
k_max = (Gamma(6) / Gamma(4.5)) / (Gamma(6) / Gamma(4.5) - Gamma(4) / Gamma(2.5))
print(f'analytic limit: r_a / b >= {1 / np.sqrt(k_max):.4f}')

Q_grid = np.linspace(0.01, 0.999, 200) * Psi0
fig, ax = plt.subplots(figsize=(7, 4.5))
sign_at_centre = {}
for r_a in [0.70, 0.75, 0.80, 1.0, 2.0]:
    fQ = eddington(plummer, plummer, r_a=r_a)
    vals = np.array([fQ(Q) for Q in Q_grid]) / f_exact(Q_grid)      # divide by the isotropic DF for display
    sign_at_centre[r_a] = np.sign(fQ(0.999 * Psi0))
    ax.plot(Q_grid / Psi0, vals, label=f'$r_a = {r_a}\\,b$')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('$Q / \\Psi(0)$'); ax.set_ylabel('$f(Q)$ / isotropic $f(\\mathcal{E}=Q)$'); ax.set_ylim(-1, 5); ax.legend()
plt.show()
print('sign of f near the centre:', sign_at_centre)
assert sign_at_centre[0.70] < 0 and sign_at_centre[0.80] > 0

**Figure 8.4.** Osipkov–Merritt DFs $f(Q)$ of the Plummer sphere for anisotropy radii $r_a$ = 0.7–2 $b$, divided by the isotropic DF at the same energy, against $Q/\Psi(0)$. For $r_a < 0.75\,b$ the DF becomes negative near $Q = \Psi(0)$.

Figure 8.4 shows that below $r_a = 0.75\,b$ the DF is negative for the most bound stars: no such model exists. Above it, AGAMA builds and samples
the model as before (`r_a=` in `QuasiSpherical`). Check that the sample has the intended anisotropy.

In [ ]:
r_a = 1.0
xv_om, _ = agama.GalaxyModel(plummer, agama.DistributionFunction(
    type='QuasiSpherical', potential=plummer, density=plummer, r_a=r_a)).sample(N)
r = np.linalg.norm(xv_om[:, :3], axis=1)
v_r = np.sum(xv_om[:, :3] * xv_om[:, 3:], axis=1) / r
n = np.histogram(r, edges)[0]
sr2 = np.histogram(r, edges, weights=v_r**2)[0] / n
st2 = (np.histogram(r, edges, weights=np.sum(xv_om[:, 3:]**2, axis=1))[0] / n - sr2) / 2
beta_s = 1 - st2 / sr2

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogx(rc, beta_s, 'o', label='AGAMA sample')
ax.semilogx(rc, rc**2 / (rc**2 + r_a**2), 'k-', label='$r^2/(r^2 + r_a^2)$')
ax.set_xlabel('$r$ [kpc]'); ax.set_ylabel('$\\beta$'); ax.legend(); plt.show()
well_sampled = n > 1000
print(f'max |beta_sample - beta_OM| in bins with > 1000 stars: '
      f'{np.max(np.abs(beta_s - rc**2 / (rc**2 + r_a**2))[well_sampled]):.3f}')
assert np.max(np.abs(beta_s - rc**2 / (rc**2 + r_a**2))[well_sampled]) < 0.05

**Figure 8.5.** Anisotropy $\beta(r)$ of a $10^5$-star AGAMA sample of the Osipkov–Merritt Plummer model with $r_a = b$ (points) and the expected $\beta = r^2/(r^2 + r_a^2)$ (line).

## Summary

- For a given density in a given potential there is exactly one isotropic DF $f(\mathcal E)$, given by Eddington's
  formula. It may be negative, in which case no isotropic model exists.
- A DF is checked by integrating it over velocities to recover the density.
- Sampling: inverse-transform for radii, rejection for speeds, random directions — or `GalaxyModel(pot, df).sample(N)`.
- Velocities drawn from Gaussians with the right dispersion are not an equilibrium: the shape of the velocity
  distribution matters, not only its moments.
- Osipkov–Merritt DFs $f(Q)$ add radial anisotropy; for the Plummer sphere they are physical only for $r_a \ge 3b/4$.

| task | AGAMA |
|---|---|
| isotropic or anisotropic spherical DF | `agama.DistributionFunction(type='QuasiSpherical', potential=, density=, r_a= or beta0=)` |
| value of a DF at phase-space points | `df(agama.ActionFinder(pot)(xv))` — DFs take actions |
| sample $N$ particles | `xv, m = agama.GalaxyModel(pot, df).sample(N)` |

## Exercises

1. **Hernquist sampling.** Sample a Hernquist sphere of scale radius $b$ by hand: invert $M(r)/M = r^2/(r+b)^2$ for radii, and use
   `eddington()` with rejection for speeds (find the maximum of $v^2 f$ numerically). Check the virial ratio.
2. **A tracer in a halo.** Use `eddington(halo, stars)` with the NFW halo and Plummer stars of Chapter 7. Recover the
   stars' density from the DF.
3. **A negative DF.** Try a cored tracer ($\rho \propto (1 + r^2/a^2)^{-1}$ with a small core radius $a$) in a point-mass potential. Where is
   the Eddington DF negative?
4. **Equilibrium in an N-body sense.** Repeat §8.3 with $N = 1000$ stars. Are the Lagrange radii of the Eddington
   sample still constant? How large are the fluctuations, and how do they scale with $N$?

## References

- A. S. Eddington (1916), MNRAS 76, 572: https://ui.adsabs.harvard.edu/abs/1916MNRAS..76..572E
- L. P. Osipkov (1979), Soviet Astronomy Letters 5, 42: https://ui.adsabs.harvard.edu/abs/1979SvAL....5...42O
- D. Merritt (1985), AJ 90, 1027: https://ui.adsabs.harvard.edu/abs/1985AJ.....90.1027M
- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §4.3.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, §5.6: https://galaxiesbook.org/chapters/I-04.-Equilibria-of-Collisionless-Stellar-Systems.html
- E. Vasiliev (2019), AGAMA, MNRAS 482, 1525: https://doi.org/10.1093/mnras/sty2672